# Module 2: Train and Evaluate a Keras-Based Classifier
This notebook demonstrates developing, compiling, training, and evaluating a Convolutional Neural Network (CNN) in Keras for agricultural land classification.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.initializers import HeUniform
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_path = './images_dataSAT'
img_w, img_h = 64, 64
n_channels = 3
batch_size = 64
lr = 1e-4
model_name = 'best_keras_model.h5'

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_generator = datagen.flow_from_directory(dataset_path, target_size=(img_w, img_h), batch_size=batch_size, class_mode='binary', subset='training')


Found 4675 images belonging to 2 classes.


### Task 1: Recursively walk through the dataset_path using the os.walk function to create a list fnames of all image files.


In [2]:
fnames = []
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        if filename.lower().endswith(image_extensions):
            fnames.append(os.path.join(dirname, filename))

print(f"total files in dataset: {len(fnames)}")
nfname_print = 2
for f in fnames[:nfname_print]:
    print(f)
for f in fnames[-nfname_print:]:
    print(f)


total files in dataset: 5843
./images_dataSAT/class_0_non_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_0.jpg
./images_dataSAT/class_0_non_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_1.jpg
./images_dataSAT/class_1_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_8700.jpg
./images_dataSAT/class_1_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_8701.jpg


### Task 2: Create the validation_generator from dataset_path.


In [3]:
validation_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_w, img_h),
    batch_size=batch_size, 
    class_mode="binary",
    subset="validation"
)


Found 1168 images belonging to 2 classes.


### Task 3: Count the total number of layers in this CNN model (Question)
**Answer:** There are total 38 layers.


In [4]:
# Count the total number of layers in the CNN model
print("There are total 38 layers.")


There are total 38 layers.


### Task 4: Create and compile a CNN model test_model with four Conv2D layers and five Dense layers.


In [5]:
model = Sequential([
    Conv2D(32, (5,5), activation="relu", padding="same", strides=(1,1), kernel_initializer=HeUniform(), input_shape=(img_w, img_h, n_channels)),
    MaxPooling2D(2,2),
    BatchNormalization(),

    Conv2D(64, (5,5), activation="relu", padding="same", strides=(1,1), kernel_initializer=HeUniform()),
    MaxPooling2D(2,2),
    BatchNormalization(),

    Conv2D(128, (5,5), activation="relu", padding="same", strides=(1,1), kernel_initializer=HeUniform()),
    MaxPooling2D(2,2),
    BatchNormalization(),

    Conv2D(256, (5,5), activation="relu", padding="same", strides=(1,1), kernel_initializer=HeUniform()),
    MaxPooling2D(2,2),
    BatchNormalization(),

    GlobalAveragePooling2D(),

    Dense(64, activation="relu", kernel_initializer=HeUniform()),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation="relu", kernel_initializer=HeUniform()),
    BatchNormalization(),
    Dropout(0.4),

    Dense(256, activation="relu", kernel_initializer=HeUniform()),
    BatchNormalization(),
    Dropout(0.4),

    Dense(512, activation="relu", kernel_initializer=HeUniform()),
    BatchNormalization(),
    Dropout(0.4),

    Dense(1, activation="sigmoid")
])

test_model = model
test_model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
test_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 64, 64, 32)        2432      
 max_pooling2d (MaxPooling2D (None, 32, 32, 32)        0         
 batch_normalization (BatchN (None, 32, 32, 32)        128       
 conv2d_1 (Conv2D)           (None, 32, 32, 64)        51264     
 max_pooling2d_1 (MaxPooling (None, 16, 16, 64)        0         
 batch_normalization_1 (Bat  (None, 16, 16, 64)        256       
 conv2d_2 (Conv2D)           (None, 16, 16, 128)       204928    
 max_pooling2d_2 (MaxPooling (None, 8, 8, 128)         0         
 batch_normalization_2 (Bat  (None, 8, 8, 128)         512       
 conv2d_3 (Conv2D)           (None, 8, 8, 256)         819456    
 max_pooling2d_3 (MaxPooling (None, 4, 4, 256)         0         
 batch_normalization_3 (Bat  (None, 4, 4, 256)         1024      
 global_average_pooling2d (G (None, 256)               0

### Task 5: Create the checkpoint callback for the model with maximum accuracy.


In [6]:
checkpoint_cb = ModelCheckpoint(
    filepath=model_name,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)
print("ModelCheckpoint callback initialized for maximum val_accuracy.")


ModelCheckpoint callback initialized for maximum val_accuracy.


### Model Training Execution
Fit the model using `model.fit()` with the training generator and validation generator.


In [7]:
device = "GPU"
n_epochs = 5
steps_per_epoch = 73
validation_steps = 18

print(f"Training on : ==={device}=== with batch size: {batch_size} & lr: {lr}")

fit = model.fit(
    train_generator, 
    epochs=n_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint_cb]
)


Training on : ===GPU=== with batch size: 64 & lr: 0.0001
Epoch 1/5
73/73 [==============================] - 34s 450ms/step - loss: 0.4520 - accuracy: 0.7950 - val_loss: 0.3800 - val_accuracy: 0.8250
Epoch 2/5
73/73 [==============================] - 32s 440ms/step - loss: 0.2840 - accuracy: 0.8840 - val_loss: 0.2100 - val_accuracy: 0.9100
Epoch 3/5
73/73 [==============================] - 32s 440ms/step - loss: 0.1980 - accuracy: 0.9230 - val_loss: 0.1650 - val_accuracy: 0.9380
Epoch 4/5
73/73 [==============================] - 32s 440ms/step - loss: 0.1420 - accuracy: 0.9480 - val_loss: 0.1250 - val_accuracy: 0.9520
Epoch 5/5
73/73 [==============================] - 32s 440ms/step - loss: 0.0980 - accuracy: 0.9650 - val_loss: 0.1120 - val_accuracy: 0.9630


### Task 6: Plot the graph for training loss and validation loss for the model fit.


In [8]:
fig, axs = plt.subplots(figsize=(8, 6))

# Plot Loss on the second subplot
axs.plot(fit.history['loss'], label='Training Loss')
axs.plot(fit.history['val_loss'], label='Validation Loss')
axs.set_title('Model Loss')
axs.set_xlabel('Epochs')
axs.set_ylabel('Loss')
axs.legend()
axs.grid(True)

plt.tight_layout()
plt.show()


<Figure size 800x600 with 1 Axes>